# **Entrenamiento de la CNN con kernels CUDA propios**

Primero, se prepara el entorno GPU y se recuperan los tensores desde Drive. Luego,
se ejecutan nuestros kernels para entrenar la misma CNN que en CPU y se evalua
validacion. Finalmente, se respaldan pesos y resultados. La particion test no se
evalua. Esta ejecucion inicial comprueba el flujo; los experimentos formales de
comparacion y las pruebas independientes corresponden al paquete de Daniela.

Se selecciona una GPU NVIDIA en Colab y se comienza con una epoca. Todavia falta
verificar la compilacion y ejecucion integrada de estos kernels en hardware real.


## **Preparar o actualizar el repositorio**

Se clona si la carpeta no existe o se actualizan las referencias si ya existe.
Se muestra el commit utilizado. Si cambian modulos Python ya importados, se
reinicia la sesion antes de continuar.


In [ ]:
from pathlib import Path
import subprocess

url_repositorio = "https://github.com/Bamo0507/Proyecto_3_Cuda_CNN.git"
revision_repositorio = "origin/main"
directorio_proyecto = Path("/content/Proyecto_3_Cuda_CNN")

if directorio_proyecto.exists():
    subprocess.run(["git", "fetch", "origin"], cwd=directorio_proyecto, check=True)
else:
    subprocess.run(["git", "clone", url_repositorio, str(directorio_proyecto)], check=True)

subprocess.run(
    ["git", "checkout", "--detach", revision_repositorio],
    cwd=directorio_proyecto,
    check=True,
)
commit_ejecutado = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=directorio_proyecto, text=True
).strip()
print("Commit ejecutado:", commit_ejecutado)


## **Instalar las dependencias de Colab**

Se ejecuta al preparar el entorno o cuando cambian las dependencias. Se utiliza
el archivo del repositorio y se muestra la salida de la instalacion. Si Colab
solicita reiniciar la sesion, se reinicia y se continua desde la verificacion del
entorno, sin volver a clonar la carpeta existente.


In [ ]:
from pathlib import Path
import subprocess
import sys

directorio_proyecto = Path("/content/Proyecto_3_Cuda_CNN")
resultado_instalacion = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     str(directorio_proyecto / "requirements_colab.txt")],
    capture_output=True,
    text=True,
)
print(resultado_instalacion.stdout, end="")
if resultado_instalacion.stderr:
    print(resultado_instalacion.stderr, end="")
resultado_instalacion.check_returncode()


## **Verificar la GPU y las versiones**

Se ejecuta despues de instalar las dependencias o reiniciar la sesion. Se registran
las versiones y el commit, y se comprueba que Numba detecte una GPU real. Esta
comprobacion todavia no demuestra que un kernel propio compile correctamente.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import subprocess
import sys

from numba import cuda

directorio_proyecto = Path("/content/Proyecto_3_Cuda_CNN")
commit_ejecutado = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=directorio_proyecto, text=True
).strip()
print("Python:", sys.version)
print("Commit:", commit_ejecutado)
for nombre_paquete in ("numpy", "numba", "numba-cuda"):
    print(nombre_paquete, version(nombre_paquete))

resultado_monitor_gpu = subprocess.run(
    ["nvidia-smi"], capture_output=True, text=True
)
print(resultado_monitor_gpu.stdout, end="")
if resultado_monitor_gpu.stderr:
    print(resultado_monitor_gpu.stderr, end="")
resultado_monitor_gpu.check_returncode()

if cuda.config.ENABLE_CUDASIM:
    raise RuntimeError("El simulador no sirve para verificar la GPU real de Colab.")
if not cuda.is_available():
    raise RuntimeError("Numba no detecta CUDA. Revisa la GPU y las dependencias.")
cuda.detect()
print("GPU detectada. Continua con la prueba del kernel propio.")


## **Montar Drive y localizar los tensores**

Se ejecuta antes de cargar los datos y se autoriza la cuenta de cada integrante.
La carpeta `proyecto_senias` debe estar en Mi unidad o tener un acceso directo alli
con ese nombre. Si el archivo no aparece, se ajusta la ruta antes de continuar.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

ruta_tensores_drive = Path(
    "/content/drive/MyDrive/proyecto_senias/02_tensores.npz"
)
print("Archivo encontrado:", ruta_tensores_drive.is_file())
if not ruta_tensores_drive.is_file():
    raise FileNotFoundError(f"No se encontraron los tensores en {ruta_tensores_drive}")


## **Copiar y validar los datos**

Luego, se copia el archivo al disco temporal de Colab para poder trabajar sin
leer desde Drive durante el entrenamiento. `compartido/datos.py` verifica imagenes, etiquetas
y clases, conservando las particiones existentes de entrenamiento, validacion y prueba.


In [ ]:
from pathlib import Path
import shutil
import sys

directorio_proyecto = Path("/content/Proyecto_3_Cuda_CNN")
sys.path.insert(0, str(directorio_proyecto / "src"))

import config
from compartido.datos import cargar_datos

config.DIRECTORIO_DATOS_PROCESADOS.mkdir(parents=True, exist_ok=True)
shutil.copyfile(ruta_tensores_drive, config.RUTA_TENSORES)

datos_imagenes = cargar_datos()
for particion in ("train", "val", "test"):
    cantidad_imagenes = len(datos_imagenes[f"x_{particion}"])
    print(f"{particion}: {cantidad_imagenes} imagenes")


## **Entrenar la CNN en GPU**

Se comienza con una epoca, batch 64 y una inicializacion comun con CPU. Se cambia
el nombre de la corrida para conservar ejecuciones anteriores. El script calienta
con pesos descartables y registra reserva/copia inicial, compilacion, entrenamiento,
recuperacion final de pesos y validacion por separado. El entrenamiento incluye las
transferencias de los minibatches y la recuperacion de metricas. La salida se muestra
mientras avanza el proceso. La comparacion formal debe usar fronteras de tiempo
iguales y no solo el tiempo de lanzamiento de kernels.


In [ ]:
from pathlib import Path
import subprocess
import sys

import config

nombre_ejecucion = "gpu_01"
cantidad_epocas = 1
tamano_lote = config.TAMANO_LOTE
directorio_proyecto = Path("/content/Proyecto_3_Cuda_CNN")
ruta_modelo_gpu = config.DIRECTORIO_MODELOS / f"{nombre_ejecucion}.npz"
ruta_registro_gpu = config.DIRECTORIO_RESULTADOS / f"{nombre_ejecucion}.json"

comando_entrenamiento = [
    sys.executable, "-u", str(directorio_proyecto / "src/entrenar_gpu.py"),
    "--epocas", str(cantidad_epocas),
    "--tamano-lote", str(tamano_lote),
    "--ruta-tensores", str(config.RUTA_TENSORES),
    "--ruta-modelo", str(ruta_modelo_gpu),
    "--ruta-registro", str(ruta_registro_gpu),
]
with subprocess.Popen(
    comando_entrenamiento,
    cwd=directorio_proyecto,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as proceso_entrenamiento:
    for linea_salida in proceso_entrenamiento.stdout:
        print(linea_salida, end="", flush=True)
    codigo_salida = proceso_entrenamiento.wait()
    if codigo_salida != 0:
        raise subprocess.CalledProcessError(codigo_salida, comando_entrenamiento)


## **Respaldar pesos y resultados en Drive**

Se ejecuta cuando el entrenamiento y el guardado terminan correctamente. Se copian
los pesos y el registro a `proyecto_senias/resultados/gpu_01`, usando el nombre de
la corrida. Los pesos se pueden reutilizar para prediccion; no incluyen los momentos
de Adam para reanudar el entrenamiento exactamente.


In [ ]:
import shutil

directorio_respaldo = ruta_tensores_drive.parent / "resultados" / nombre_ejecucion
if not ruta_modelo_gpu.is_file() or not ruta_registro_gpu.is_file():
    raise FileNotFoundError("Faltan los archivos de una ejecucion terminada.")
directorio_respaldo.mkdir(parents=True, exist_ok=False)

for ruta_resultado in (ruta_modelo_gpu, ruta_registro_gpu):
    ruta_respaldo = directorio_respaldo / ruta_resultado.name
    shutil.copyfile(ruta_resultado, ruta_respaldo)
    print("Respaldado:", ruta_respaldo)
